# 04 — Stacks, Queues, and Linear Abstract Data Types

## Theoretical Foundations of Linear Abstract Data Types

An **Abstract Data Type (ADT)** defines a mathematical model for data types characterized by specific behaviors and operational contracts from the point of view of a user, independent of implementation details.

Two foundational linear ADTs are:
1. **Stack (LIFO — Last-In, First-Out):**
   - Elements are inserted and removed from the same end, termed the **top**.
   - Core operations: `push(item)`, `pop() -> item`, `peek() -> item`, `is_empty() -> bool`.
   - Applications: Call stack execution, depth-first search (DFS), undo-redo buffers, syntax and bracket validation.
2. **Queue (FIFO — First-In, First-Out):**
   - Elements are inserted at the **rear** (enqueue) and removed from the **front** (dequeue).
   - Core operations: `enqueue(item)`, `dequeue() -> item`, `front() -> item`, `is_empty() -> bool`.
   - Applications: Task scheduling, breadth-first search (BFS), streaming buffers, asynchronous messaging queues.

## Implementation Tradeoffs in Python

- **List as Stack:** `list.append()` and `list.pop()` operate at the end of a dynamic array in amortized $O(1)$ time. Python lists are highly efficient stacks.
- **List as Queue:** While `list.append()` enqueues in $O(1)$, removing from the front via `list.pop(0)` requires shifting all remaining $n-1$ elements in memory, causing an **$O(n)$ performance bottleneck**.
- **`collections.deque`:** Implemented as a doubly linked list of fixed-size blocks (64 elements per block). It provides **guaranteed $O(1)$ operations at both ends** (`append`, `pop`, `appendleft`, `popleft`).


In [ ]:
# 1. Object-Oriented Array-Backed Stack Implementation
from typing import Generic, TypeVar

T = TypeVar("T")


class ArrayStack(Generic[T]):
    """LIFO Stack backed by a dynamic array (Python list)."""

    def __init__(self) -> None:
        self._buffer: list[T] = []

    def push(self, item: T) -> None:
        """Pushes an element onto the top of the stack in amortized O(1) time."""
        self._buffer.append(item)

    def pop(self) -> T:
        """Removes and returns the top element. Raises IndexError if empty."""
        if self.is_empty():
            raise IndexError("pop from empty stack")
        return self._buffer.pop()

    def peek(self) -> T:
        """Returns the top element without removing it. Raises IndexError if empty."""
        if self.is_empty():
            raise IndexError("peek on empty stack")
        return self._buffer[-1]

    def is_empty(self) -> bool:
        return len(self._buffer) == 0

    def __len__(self) -> int:
        return len(self._buffer)

    def __repr__(self) -> str:
        return f"ArrayStack(top -> {self._buffer[::-1]})"


# Test suite
stack: ArrayStack[int] = ArrayStack()
stack.push(10)
stack.push(20)
stack.push(30)

print(f"Stack state: {stack}")
assert len(stack) == 3
assert stack.peek() == 30
assert stack.pop() == 30
assert stack.pop() == 20
assert stack.pop() == 10
assert stack.is_empty()

## Linked List-Based Stack Implementation

To illustrate pointer-based dynamic memory structures without contiguous array over-allocation, we implement a singly-linked stack where each `Node` maintains a payload and a pointer to the next lower node. All push and pop operations are **strictly $O(1)$ worst-case**.


In [ ]:
# 2. Linked List-Based Stack
class StackNode(Generic[T]):
    """Node element in a singly linked stack."""

    def __init__(self, data: T, next_node: "StackNode[T] | None" = None) -> None:
        self.data = data
        self.next = next_node


class LinkedStack(Generic[T]):
    """Singly-linked pointer stack with strict O(1) push and pop."""

    def __init__(self) -> None:
        self._top: StackNode[T] | None = None
        self._size = 0

    def push(self, item: T) -> None:
        self._top = StackNode(item, next_node=self._top)
        self._size += 1

    def pop(self) -> T:
        if self._top is None:
            raise IndexError("pop from empty linked stack")
        value = self._top.data
        self._top = self._top.next
        self._size -= 1
        return value

    def peek(self) -> T:
        if self._top is None:
            raise IndexError("peek on empty linked stack")
        return self._top.data

    def is_empty(self) -> bool:
        return self._top is None

    def __len__(self) -> int:
        return self._size


linked_stack: LinkedStack[str] = LinkedStack()
linked_stack.push("input_tensor")
linked_stack.push("hidden_activation")
linked_stack.push("output_logits")

assert len(linked_stack) == 3
assert linked_stack.pop() == "output_logits"
assert linked_stack.peek() == "hidden_activation"
print("Linked stack operations successfully verified.")

## FIFO Queue Implementation: `collections.deque` and Circular Buffers

For queues, using `collections.deque` avoids the $O(n)$ shifting penalty of lists. Alternatively, a **circular buffer** uses a fixed-size array with modular arithmetic pointers `head = (head + 1) % capacity` to provide $O(1)$ enqueue and dequeue within a bounded footprint.


In [ ]:
# 3. Queue Implementations: DequeQueue and CircularQueue
from collections import deque


class DequeQueue(Generic[T]):
    """FIFO Queue backed by collections.deque for O(1) operations."""

    def __init__(self) -> None:
        self._container: deque[T] = deque()

    def enqueue(self, item: T) -> None:
        self._container.append(item)

    def dequeue(self) -> T:
        if self.is_empty():
            raise IndexError("dequeue from empty queue")
        return self._container.popleft()

    def front(self) -> T:
        if self.is_empty():
            raise IndexError("front on empty queue")
        return self._container[0]

    def is_empty(self) -> bool:
        return len(self._container) == 0

    def __len__(self) -> int:
        return len(self._container)


class CircularQueue(Generic[T]):
    """Bounded Circular Array Queue with modular index wrapping."""

    def __init__(self, capacity: int) -> None:
        if capacity <= 0:
            raise ValueError("Capacity must be positive")
        self._capacity = capacity
        self._buffer: list[T | None] = [None] * capacity
        self._head = 0
        self._tail = 0
        self._size = 0

    def enqueue(self, item: T) -> None:
        if self._size == self._capacity:
            raise OverflowError("enqueue into full circular queue")
        self._buffer[self._tail] = item
        self._tail = (self._tail + 1) % self._capacity
        self._size += 1

    def dequeue(self) -> T:
        if self.is_empty():
            raise IndexError("dequeue from empty circular queue")
        item = self._buffer[self._head]
        self._buffer[self._head] = None
        self._head = (self._head + 1) % self._capacity
        self._size -= 1
        return item  # type: ignore[return-value]

    def is_empty(self) -> bool:
        return self._size == 0

    def __len__(self) -> int:
        return self._size


# Testing both queue variants
d_queue: DequeQueue[str] = DequeQueue()
d_queue.enqueue("sample_1")
d_queue.enqueue("sample_2")
assert d_queue.dequeue() == "sample_1"
assert d_queue.dequeue() == "sample_2"

c_queue: CircularQueue[int] = CircularQueue(capacity=3)
c_queue.enqueue(100)
c_queue.enqueue(200)
c_queue.enqueue(300)
assert c_queue.dequeue() == 100
c_queue.enqueue(400)  # Wraps around index 0
assert c_queue.dequeue() == 200
assert c_queue.dequeue() == 300
assert c_queue.dequeue() == 400
assert c_queue.is_empty()
print("Queue variants successfully validated.")

## Algorithmic Application: Bracket Balancer and Syntax Validator

A classic stack application is validating balanced delimiters in source code and mathematical expressions: parentheses `()`, brackets `[]`, and braces `{}`.

### Invariant
Every closing delimiter must match the most recently encountered unclosed opening delimiter (LIFO order).


In [ ]:
# 4. Balanced Delimiter Validation Algorithm
def validate_balanced_delimiters(expression: str) -> bool:
    """Validates whether all bracket delimiters in expression are symmetrically balanced."""
    matching_pairs = {")": "(", "]": "[", "}": "{"}
    opening_delimiters = set(matching_pairs.values())
    delimiter_stack: list[str] = []

    for char in expression:
        if char in opening_delimiters:
            delimiter_stack.append(char)
        elif char in matching_pairs and (
            not delimiter_stack or delimiter_stack.pop() != matching_pairs[char]
        ):
            return False

    return len(delimiter_stack) == 0


# Test scenarios
valid_expressions = [
    "{[()()]}",
    "def forward(x: Tensor) -> Tensor: return self.fc(self.act(x))",
    "loss = (y_true - y_pred) ** 2",
]

invalid_expressions = [
    "{[([)]}",  # Crossing brackets
    "((())",  # Unclosed opening bracket
    "())(",  # Misordered
]

for expr in valid_expressions:
    assert validate_balanced_delimiters(expr), f"Failed on valid: {expr}"

for expr in invalid_expressions:
    assert not validate_balanced_delimiters(expr), f"Failed on invalid: {expr}"

print("Syntax balancer validated on all test expressions.")